# Section 3 — The Discretization Induced by the Forest

This notebook tests the implementation of **Section 3**:
1. **Forest representation (Section 2)**: `Tree` and `Forest` structures.
2. **Definition 1 (Cells of a feature)**: collecting and sorting thresholds $\ell_0 < \ell_1 < \dots < \ell_{k-1}$ per feature into intervals $(-\infty, \ell_0], (\ell_0, \ell_1], \dots, (\ell_{k-1}, +\infty)$.
3. **Definition 2 (Cell-count function)**: computing $\text{cells}(f) = k + 1$.
4. **Proposition 1 (Lossless discretization)**: mapping $\mathbb{R}^F \to \mathbb{N}^F$ and verifying that classification on cell indices matches real-valued classification.

In [1]:
import os
import sys
import numpy as np

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from rfbdd.forest import Tree, Forest
from rfbdd.discretize import Discretization, discretize

def check(condition: bool, message: str) -> None:
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {message}")
    assert condition, message

print("Setup complete.")

Setup complete.


## 3.1 Sample Forest Construction (Section 2)

We build a forest with $D = 2$ trees over $F = 2$ features and $|L| = 2$ classes:
- **Tree $T_0$**:
  - Node `0` (root): tests `feature 0 <= 5.7` $\rightarrow$ left leaf `1` (class `0`), right node `2`
  - Node `2`: tests `feature 1 <= 10.0` $\rightarrow$ left leaf `3` (class `0`), right leaf `4` (class `1`)
- **Tree $T_1$**:
  - Node `0` (root): tests `feature 0 <= 2.5` $\rightarrow$ left leaf `1` (class `0`), right node `2`
  - Node `2`: tests `feature 0 <= 8.1` $\rightarrow$ left leaf `3` (class `1`), right leaf `4` (class `0`)

In [2]:
t0 = Tree(
    feature=np.array([0, -1, 1, -1, -1], dtype=np.int64),
    threshold=np.array([5.7, np.nan, 10.0, np.nan, np.nan], dtype=np.float64),
    left=np.array([1, -1, 3, -1, -1], dtype=np.int64),
    right=np.array([2, -1, 4, -1, -1], dtype=np.int64),
    label=np.array([-1, 0, -1, 0, 1], dtype=np.int64),
)

t1 = Tree(
    feature=np.array([0, -1, 0, -1, -1], dtype=np.int64),
    threshold=np.array([2.5, np.nan, 8.1, np.nan, np.nan], dtype=np.float64),
    left=np.array([1, -1, 3, -1, -1], dtype=np.int64),
    right=np.array([2, -1, 4, -1, -1], dtype=np.int64),
    label=np.array([-1, 0, -1, 1, 0], dtype=np.int64),
)

forest = Forest(
    trees=[t0, t1],
    n_features=2,
    n_classes=2,
    classes=np.array([0, 1]),
    name="toy_forest",
)

disc = discretize(forest)
print(f"Forest '{forest.name}': D={forest.D} trees, F={forest.n_features} features, W={forest.W}")

Forest 'toy_forest': D=2 trees, F=2 features, W=2


## 3.2 Threshold Extraction and Cell Count (Definitions 1 & 2)

Verify that:
- Feature `0` has sorted thresholds `[2.5, 5.7, 8.1]` and $\text{cells}(0) = 4$.
- Feature `1` has threshold `[10.0]` and $\text{cells}(1) = 2$.
- The intervals of feature `0` are $(-\infty, 2.5], (2.5, 5.7], (5.7, 8.1], (8.1, +\infty)$.

In [3]:
check(np.array_equal(disc.thresholds[0], np.array([2.5, 5.7, 8.1])), "Feature 0 thresholds are [2.5, 5.7, 8.1]")
check(np.array_equal(disc.thresholds[1], np.array([10.0])), "Feature 1 threshold is [10.0]")

check(disc.cells(0) == 4 and disc.cells(1) == 2, "cells(0) == 4 and cells(1) == 2")

expected_bounds_f0 = [(-np.inf, 2.5), (2.5, 5.7), (5.7, 8.1), (8.1, np.inf)]
for c, exp in enumerate(expected_bounds_f0):
    bounds = disc.cell_bounds(0, c)
    check(bounds == exp, f"Feature 0 cell {c} interval: ({bounds[0]}, {bounds[1]}]")

[PASS] Feature 0 thresholds are [2.5, 5.7, 8.1]
[PASS] Feature 1 threshold is [10.0]
[PASS] cells(0) == 4 and cells(1) == 2
[PASS] Feature 0 cell 0 interval: (-inf, 2.5]
[PASS] Feature 0 cell 1 interval: (2.5, 5.7]
[PASS] Feature 0 cell 2 interval: (5.7, 8.1]
[PASS] Feature 0 cell 3 interval: (8.1, inf]


## 3.3 Mapping Real Values to Cell Indices

Each interval (except the last) is left-open and right-closed $(\ell_{c-1}, \ell_c]$, so a value equal to $\ell_c$ falls into cell $c$, while any value strictly greater than $\ell_c$ falls into cell $c+1$.

In [4]:
test_points_f0 = [
    (-1.0, 0),
    (2.5, 0),
    (2.5001, 1),
    (5.7, 1),
    (7.0, 2),
    (8.1, 2),
    (9.0, 3),
]

for val, exp_cell in test_points_f0:
    c = disc.cell_index(0, val)
    check(c == exp_cell, f"feature 0, value {val:6.4f} -> cell {c} (expected {exp_cell})")

sample = np.array([7.0, 10.1])
sample_cells = disc.cells_of(sample)
check(np.array_equal(sample_cells, np.array([2, 1])), f"Sample {sample.tolist()} -> cell indices {sample_cells.tolist()}")

[PASS] feature 0, value -1.0000 -> cell 0 (expected 0)
[PASS] feature 0, value 2.5000 -> cell 0 (expected 0)
[PASS] feature 0, value 2.5001 -> cell 1 (expected 1)
[PASS] feature 0, value 5.7000 -> cell 1 (expected 1)
[PASS] feature 0, value 7.0000 -> cell 2 (expected 2)
[PASS] feature 0, value 8.1000 -> cell 2 (expected 2)
[PASS] feature 0, value 9.0000 -> cell 3 (expected 3)
[PASS] Sample [7.0, 10.1] -> cell indices [2, 1]


## 3.4 Proposition 1 (Lossless Discretization)

Verify that:
1. `taus` maps each internal node to its corresponding threshold cell index $\tau$.
2. Across a grid of real samples $(x_0, x_1) \in \mathbb{R}^2$, each tree $T_t$ and the full forest $\mathcal{F}$ classify identically using real values (`tree.decide(x)`) and cell indices (`tree.decide_cells(disc.cells_of(x), disc.taus[t])`).

In [5]:
expected_taus = [{0: 1, 2: 0}, {0: 0, 2: 2}]
check(disc.taus == expected_taus, f"Internal node tau mapping matches: {disc.taus}")

xs_f0 = [-5.0, 2.5, 4.0, 5.7, 6.8, 8.1, 12.0]
xs_f1 = [0.0, 10.0, 15.0]

all_match = True
for v0 in xs_f0:
    for v1 in xs_f1:
        x = np.array([v0, v1])
        c = disc.cells_of(x)
        for t_idx, tree in enumerate(forest.trees):
            if tree.decide(x) != tree.decide_cells(c, disc.taus[t_idx]):
                all_match = False
        if not np.array_equal(forest.votes(x), forest.votes_cells(c, disc.taus)):
            all_match = False

check(all_match, "Proposition 1 verified: real-valued and cell-indexed classifications match on all samples")

[PASS] Internal node tau mapping matches: [{0: 1, 2: 0}, {0: 0, 2: 2}]
[PASS] Proposition 1 verified: real-valued and cell-indexed classifications match on all samples
